# SCOS Snooker Ball Detection — Train on Kaggle (Free GPU)

This notebook trains a YOLOv8 model for snooker/pool ball detection using Kaggle's free GPU (30 hrs/week).

## Workflow
1. Install Ultralytics + Hugging Face Hub
2. Load dataset (upload as Kaggle Dataset or download from HF Hub)
3. Train YOLOv8s (or YOLOv11s)
4. Validate on test set
5. Export to ONNX
6. Push model to Hugging Face Hub (free, permanent storage)

## Before you start
- Set **Accelerator** to **GPU (T4 x2 or P100)** in Kaggle notebook settings
- Add your dataset via **Add Data** → upload `pool_dataset/` as a Kaggle dataset
- Set HF_TOKEN in Kaggle Secrets (Settings → Secrets → Add new secret)

In [ ]:
# Cell 1: Install dependencies
!pip install -q ultralytics huggingface_hub onnx onnxruntime
print('Dependencies installed.')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Cell 3: Configuration — EDIT THESE

# Dataset path on Kaggle (after adding as Kaggle Dataset)
# If you uploaded pool_dataset/ as a Kaggle dataset named 'scos-pool-dataset',
# it will be at /kaggle/input/scos-pool-dataset/
DATASET_PATH = '/kaggle/input/scos-pool-dataset'

# Model configuration
MODEL_SIZE = 'yolov8s.pt'  # Options: yolov8n.pt (fastest), yolov8s.pt (balanced), yolov8m.pt (accurate)
EPOCHS = 100
BATCH_SIZE = 16
IMGSZ = 640

# Hugging Face Hub — where the trained model will be pushed
# Create a model repo at: https://huggingface.co/new (Model, Object Detection)
HF_REPO_ID = 'your-username/scos-yolov8s'  # CHANGE THIS

print(f'Dataset: {DATASET_PATH}')
print(f'Model: {MODEL_SIZE}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Image size: {IMGSZ}')
print(f'HF Repo: {HF_REPO_ID}')

In [ ]:
# Cell 4: Prepare data.yaml for Kaggle paths
import os
import yaml

# Check dataset structure
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'{split}: {n_imgs} images, {n_lbls} labels')

# Read original data.yaml to get class names
orig_yaml = os.path.join(DATASET_PATH, 'data.yaml')
with open(orig_yaml, 'r') as f:
    orig_config = yaml.safe_load(f)

print(f'\nOriginal classes ({orig_config["nc"]}): {orig_config["names"]}')

# Write a Kaggle-friendly data.yaml
kaggle_yaml = {
    'path': DATASET_PATH,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': orig_config['nc'],
    'names': orig_config['names'],
}

yaml_path = '/kaggle/working/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(kaggle_yaml, f, default_flow_style=False)

print(f'\nKaggle data.yaml written to: {yaml_path}')
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
# Cell 5: Train the model
from ultralytics import YOLO

model = YOLO(MODEL_SIZE)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    project='/kaggle/working/scos_train',
    name='yolo_train',
    save=True,
    save_period=20,  # checkpoint every 20 epochs
    cache=True,      # cache images in RAM for speed
    workers=4,
    patience=30,     # early stopping
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    label_smoothing=0.0,
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    plots=True,
    verbose=True,
    seed=42,
)

print('Training complete!')

In [ ]:
# Cell 6: Validate on test set
metrics = model.val(split='test')
print(f'\nmAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:     {metrics.box.mp:.4f}')
print(f'Recall:        {metrics.box.mr:.4f}')

# Per-class results
names = model.names
for i, (ap, ar) in enumerate(zip(metrics.box.ap50, metrics.box.ar)):
    print(f'  {names[i]:>12s}: AP50={ap:.3f}  AR={r:.3f}' if False else f'  Class {i}: AP50={ap:.3f}  AR={ar:.3f}')

In [ ]:
# Cell 7: Export to ONNX for deployment
best_pt = '/kaggle/working/scos_train/yolo_train/weights/best.pt'
model = YOLO(best_pt)

onnx_path = model.export(format='onnx', imgsz=IMGSZ, simplify=True, dynamic=False)
print(f'ONNX exported: {onnx_path}')

# Also export TorchScript as backup
torchscript_path = model.export(format='torchscript')
print(f'TorchScript exported: {torchscript_path}')

In [ ]:
# Cell 8: Push model to Hugging Face Hub
# Requires HF_TOKEN in Kaggle Secrets (Settings → Secrets)
import os
from huggingface_hub import HfApi, create_repo

# Get HF token from Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')

api = HfApi(token=hf_token)

# Create repo if it doesn't exist
try:
    create_repo(HF_REPO_ID, token=hf_token, repo_type='model', exist_ok=True)
    print(f'Repo ready: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'Repo creation: {e}')

# Upload model files
weights_dir = '/kaggle/working/scos_train/yolo_train/weights'
files_to_upload = [
    ('best.pt', f'{weights_dir}/best.pt'),
    ('best.onnx', onnx_path),
    ('last.pt', f'{weights_dir}/last.pt'),
]

for filename, filepath in files_to_upload:
    if os.path.exists(filepath):
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=filename,
            repo_id=HF_REPO_ID,
            repo_type='model',
            token=hf_token,
        )
        print(f'Uploaded: {filename}')
    else:
        print(f'Skipped (not found): {filepath}')

# Upload training results plot
results_png = '/kaggle/working/scos_train/yolo_train/results.png'
if os.path.exists(results_png):
    api.upload_file(
        path_or_fileobj=results_png,
        path_in_repo='results.png',
        repo_id=HF_REPO_ID,
        repo_type='model',
        token=hf_token,
    )
    print('Uploaded: results.png')

print(f'\nModel available at: https://huggingface.co/{HF_REPO_ID}')

In [ ]:
# Cell 9: Create model card (README.md) on HF Hub
from huggingface_hub import HfApi
import yaml

model_card = f'''---
license: apache-2.0
task_categories:
- object-detection
tags:
- yolo
- snooker
- pool
- billiards
- computer-vision
library_name: ultralytics
---

# SCOS Snooker Ball Detection Model

Trained with YOLOv8 on Kaggle free GPU.

## Model Details
- **Architecture**: {MODEL_SIZE}
- **Image size**: {IMGSZ}x{IMGSZ}
- **Classes**: {orig_config['nc']}
- **Class names**: {orig_config['names']}

## Usage

### Python (Ultralytics)
```python
from ultralytics import YOLO
model = YOLO('your-username/scos-yolov8s')  # download from HF Hub
results = model('image.jpg')
```

### ONNX Runtime
```python
import onnxruntime as ort
session = ort.InferenceSession('best.onnx')
```

### SCOS Backend (Node.js)
Use the HFDetector adapter in the SCOS backend to call this model
via Hugging Face Space or local ONNX inference.
'''

# Write and upload
with open('/kaggle/working/README.md', 'w') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj='/kaggle/working/README.md',
    path_in_repo='README.md',
    repo_id=HF_REPO_ID,
    repo_type='model',
    token=hf_token,
)
print('Model card uploaded.')

In [ ]:
# Cell 10: Quick inference test
import glob
from ultralytics import YOLO

model = YOLO('/kaggle/working/scos_train/yolo_train/weights/best.pt')

# Run on a few test images
test_images = glob.glob(f'{DATASET_PATH}/test/images/*.jpg')[:5]
if not test_images:
    test_images = glob.glob(f'{DATASET_PATH}/test/images/*.png')[:5]

for img_path in test_images:
    results = model(img_path, verbose=False)
    r = results[0]
    n_dets = len(r.boxes)
    print(f'{os.path.basename(img_path)}: {n_dets} detections')
    for box in r.boxes[:5]:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        print(f'  class={cls_id} conf={conf:.3f} xyxy={[round(v,1) for v in box.xyxy[0].tolist()]}')

print('\nInference test complete!')

## Next Steps

1. **Model is now on Hugging Face Hub** at `https://huggingface.co/your-username/scos-yolov8s`
2. **Deploy to HF Space**: Copy the `hf-space/` directory from the repo to a new HF Space (Docker type)
3. **Use locally**: Download `best.pt` or `best.onnx` from HF Hub and use the `HFDetector` adapter in SCOS backend
4. **Iterate**: Re-run with more epochs, larger model, or more data as the SCOS dataset grows